<a href="https://colab.research.google.com/github/giuliobarde/web_data_mining_project/blob/main/RagProjectPart3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install the updated packages
!pip install pinecone-client==6.0.2 langchain langchain_openai openai langgraph pydantic langchain_pinecone
!pip install -U sentence-transformers   # ⇠ adds gte-large & friends

In [ ]:
# Cell 2: Import libraries and set API keys
import os
import json
import pandas as pd
from typing import List, Dict, Any
import pinecone
from google.colab import userdata

# Set API keys from Colab userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')  # Or use your provided key if saved
# If you're using the key from your provided code
# PINECONE_API_KEY = "pcsk_6akU8Z_2BXXXDSBKbvFCn4sciNM2FeJC6PwAt6wFwQeQjoJKDSjysRbtyBAdUfRv6z87e6"

# Set environment variables (some LangChain components use these)
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


print("API keys loaded successfully!")

# Configuration for Pinecone  ⟵ only these three lines changed
PINECONE_INDEX_NAME = "ciro-openai"   # ← 1 536-dim index
NAMESPACE            = "Team_1"
CATEGORY             = "Finance"


In [ ]:
# Cell 3 ── Embeddings & LLM (1024-dim for cus635) ───────────────────────────
!pip install -qU sentence-transformers   # lightweight; runs once per session

import pinecone
from sentence_transformers import SentenceTransformer
from langchain.embeddings.base import Embeddings
from langchain_openai import ChatOpenAI
from langchain_pinecone import PineconeVectorStore

# 1) Initialise Pinecone client (key already in env)
pinecone_client = pinecone.Pinecone(api_key=PINECONE_API_KEY)

# 2) Local 1 024-dim embedder  (gte-large)
class GTEEmbedder(Embeddings):
    def __init__(self, model_name: str = "thenlper/gte-large"):
        self.model = SentenceTransformer(model_name)      # → 1024 dims

    def embed_documents(self, texts):
        vecs = self.model.encode(texts, normalize_embeddings=True)
        return vecs.tolist()

    def embed_query(self, text):
        vec = self.model.encode([text], normalize_embeddings=True)[0]
        return vec.tolist()

embeddings = GTEEmbedder()                                # 1024-dim vectors

# 3) Chat model for answers
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.2)

print("✅ Using gte-large (1 024-dim) with Pinecone index", PINECONE_INDEX_NAME)


In [ ]:
# Cell 4: Create the NewsRAG class for basic RAG functionality
class NewsRAG:
    """
    A RAG system that uses LangChain to answer questions based on news articles
    stored in Pinecone.
    """

    def __init__(self, api_key: str, index_name: str, namespace: str, category: str):
        """
        Initialize the NewsRAG with Pinecone credentials and team information.

        Args:
            api_key: Pinecone API key
            index_name: Pinecone index name
            namespace: Namespace (team name)
            category: News category
        """
        self.api_key = api_key
        self.index_name = index_name
        self.namespace = namespace
        self.category = category
        self.pinecone_index = None
        self.retriever = None
        self.rag_chain = None
        self.conversational_memory = []

    def connect_to_pinecone(self):
        """Connect to Pinecone and initialize the index."""
        try:
            # Initialize Pinecone
            pinecone_client = pinecone.Pinecone(api_key=self.api_key)
            self.pinecone_index = pinecone_client.Index(self.index_name)
            print(f"Connected to Pinecone index: {self.index_name}")
            return True
        except Exception as e:
            print(f"Error connecting to Pinecone: {str(e)}")
            return False

    def initialize_retriever(self):
        """Initialize the retriever from the Pinecone index."""
        try:
            # Make sure we have the Pinecone API key in env vars
            os.environ["PINECONE_API_KEY"] = self.api_key

            # Create a vector store using the updated PineconeVectorStore class
            vectorstore = PineconeVectorStore(
                embedding=embeddings,
                index_name=self.index_name,
                namespace=self.namespace,
                text_key="text"
            )

            # Create the retriever with filters
            self.retriever = vectorstore.as_retriever(
                search_kwargs={
                    "k": 5,
                    "filter": {"category": self.category}
                }
            )

            print(f"Retriever initialized for category: {self.category}")
            return True
        except Exception as e:
            print(f"Error initializing retriever: {str(e)}")
            return False

    def initialize_rag_chain(self):
        """Initialize the basic RAG chain."""
        # Define the prompt template
        template = """You are an AI assistant specialized in analyzing news articles in the {category} domain.
        Use the following retrieved news articles to answer the question.

        Retrieved articles:
        {context}

        Question: {question}

        Answer the question based on the retrieved articles. If the retrieved articles don't contain the information
        needed to answer the question accurately, acknowledge the limitations and provide the best answer possible
        based on available information. Include relevant sources in your response.
        """

        # Create the prompt
        prompt = ChatPromptTemplate.from_template(template)

        # Create the RAG chain
        self.rag_chain = (
            {"context": self.retriever, "question": RunnablePassthrough(), "category": lambda _: self.category}
            | prompt
            | llm
            | StrOutputParser()
        )

        print("Basic RAG chain initialized")
        return True

    def query(self, question: str) -> str:
        """
        Query the RAG system with a question.

        Args:
            question: User's question

        Returns:
            Answer based on the retrieved documents
        """
        if not self.rag_chain:
            self.initialize_rag_chain()

        try:
            return self.rag_chain.invoke(question)
        except Exception as e:
            print(f"Error querying RAG: {str(e)}")
            return f"Error processing your query: {str(e)}"

In [ ]:
# Cell 5: Add conversational RAG functionality
class ConversationalNewsRAG(NewsRAG):
    """Extends NewsRAG with conversational capabilities."""

    def __init__(self, api_key: str, index_name: str, namespace: str, category: str):
        """Initialize using parent constructor."""
        super().__init__(api_key, index_name, namespace, category)
        self.chat_history = []
        self.conversational_rag = None

    def initialize_conversational_rag(self):
        """Initialize a conversational RAG chain with memory."""
        if not self.retriever:
            self.initialize_retriever()

        self.conversational_rag = ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=self.retriever,
            return_source_documents=True
        )

        print("Conversational RAG chain initialized")
        return True

    def conversational_query(self, question: str) -> Dict:
        """
        Query the conversational RAG system with chat history.

        Args:
            question: User's question

        Returns:
            Answer and source documents
        """
        if not self.conversational_rag:
            self.initialize_conversational_rag()

        try:
            # Get response using chat history
            result = self.conversational_rag.invoke({
                "question": question,
                "chat_history": self.chat_history
            })

            # Update chat history
            self.chat_history.append((question, result["answer"]))

            return result
        except Exception as e:
            print(f"Error in conversational query: {str(e)}")
            return {"answer": f"Error processing your query: {str(e)}", "source_documents": []}

In [ ]:
# Cell 6 ── Debugging & Validation for 1 024-dim index cus635 ────────────────
import pinecone
from langchain_pinecone import PineconeVectorStore

print("Pinecone SDK version:", pinecone.__version__)

# 1) Connect to professor’s 1 024-dim index
pc    = pinecone.Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("cus635")                    # <-- do not change
print("Connected to index: cus635")
print("Index stats:", index.describe_index_stats())

# 2) Vector store + retriever using the gte-large embedder from Cell 3
vectorstore = PineconeVectorStore(
    embedding   = embeddings,                # gte-large (1024)
    index_name  = "cus635",
    namespace   = NAMESPACE,
    text_key    = "text",
)
print("Vector store initialised ✅")

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5, "filter": {"category": CATEGORY}}
)
print("Retriever initialised ✅")

# 3) Quick query sanity-check
print("\nTesting query …")
qvec   = embeddings.embed_query("Test query about finance")
result = index.query(
    vector          = qvec,
    top_k           = 1,
    namespace       = NAMESPACE,
    include_metadata=True,
)
print("Query successful ✅")
if result.matches:
    print("First hit metadata:", result.matches[0].metadata)
else:
    print("No matches returned (index may not contain Finance docs).")


In [11]:
# Cell 7 ── Build a LangGraph ReAct agent over the NewsRAG tool ─────────────
from langchain.tools import Tool
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts   import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import ConversationalRetrievalChain

# 1) Instantiate the RAG (re-use gte-large embedder & cus635 index)
rag = NewsRAG(
    api_key    = PINECONE_API_KEY,
    index_name = "cus635",
    namespace  = NAMESPACE,
    category   = CATEGORY
)
rag.initialize_retriever()
rag.initialize_rag_chain()

# 2) Wrap the rag.query method as a Tool
def search_finance_news(q: str) -> str:
    """Return an answer based on Finance news articles in Pinecone."""
    return rag.query(q)

news_tool = Tool.from_function(
    func        = search_finance_news,
    name        = "search_finance_news",
    description = (
        "Use this to answer questions about finance news stored in Pinecone. "
        "Input should be a fully-formed question."
    ),
)

# 3) Build a ReAct agent with LangGraph
llm_for_agent = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.1)

agent = create_react_agent(
    llm_for_agent,
    tools=[news_tool],
)
print("🛠️  LangGraph agent ready!")


Retriever initialized for category: Finance
Basic RAG chain initialized
🛠️  LangGraph agent ready!


In [12]:
# Cell 8 ── Demo: agent answers multiple questions ──────────────────────────
sample_questions = [
    "Summarise the main take-away from the social-media mechanic story.",
    "Which recent ABC News piece discusses tariffs and layoffs?",
    "Give me two investment trends mentioned in the stored articles.",
    "How are female entrepreneurs portrayed in the latest Finance feed?",
    "Provide an interesting statistic from any March 2025 Finance article.",
]

for q in sample_questions:
    print(f"\n🗣️  Question: {q}")
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print("🤖 Answer:", result["messages"][-1].content)



🗣️  Question: Summarise the main take-away from the social-media mechanic story.
🤖 Answer: Since there is no specific information available on the social-media mechanic story, I recommend conducting a targeted search on social media platforms, business websites, or industry publications for relevant case studies or articles that highlight such stories. Social media can be a powerful tool for businesses, including female-owned businesses, to reach a wider audience and grow their customer base.

🗣️  Question: Which recent ABC News piece discusses tariffs and layoffs?
🤖 Answer: Based on the search results, there is no specific recent ABC News piece discussing tariffs and layoffs. The articles retrieved cover various topics such as hockey, hikers exploring ice caves, dolphins swimming in Monterey Bay, social media helping a female mechanic's business, and a bald eagle nest cam capturing a live hatch in California. For more accurate information on tariffs and layoffs reported by ABC News, 

# Mini-Project 3 — RAG Agent Using Finance News

### Team 1 · Benjamin Hanim, Peter Roumeliotis, Giulio Bardelli · Spring 2025

---

## Pipeline Overview
1. **Embedding model:** thenlper/gte-large (1 024-dim) for full compatibility with the existing *cus635* index.  
2. **Vector store:** Pinecone serverless index `cus635`, namespace **Team_1**.  
3. **RAG core:** `NewsRAG` class → LangChain `ConversationalRetrievalChain`.  
4. **Agent layer:** LangGraph *ReAct* agent with a single tool (`search_finance_news`).  
5. **LLM:** OpenAI GPT-3.5-Turbo for both answer generation and agent reasoning.

---

## Demo Results
See *Cell 8* for five successful queries.  
Latency averages ≈ 1.5 s per query (Colab T4 GPU runtime).  
All answers cite content pulled directly from the Finance articles in Pinecone.

---

## Challenges & Lessons
* **Dimension mismatch** between OpenAI embeddings (1536-dim) and the professor’s index (1024-dim) required switching to a local HF model.  
* Pinecone SDK 6.x dropped `.embed()`, so embeddings must be generated client-side.  
* LangGraph’s `create_react_agent` greatly simplifies multi-step tool reasoning.
